In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('GlobalWeatherRepository.csv')
print("Shape:", df.shape)
df.head()

Shape: (144432, 41)


,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,...,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,Afghanistan,Kabul,34.52,69.18,Asia/Kabul,1715849100,2024-05-16 13:15,26.6,79.8,Partly Cloudy,...,8.4,26.6,1,1,04:50 AM,06:50 PM,12:12 PM,01:11 AM,Waxing Gibbous,55
1,Albania,Tirana,41.33,19.82,Europe/Tirane,1715849100,2024-05-16 10:45,19.0,66.2,Partly cloudy,...,1.1,2.0,1,1,05:21 AM,07:54 PM,12:58 PM,02:14 AM,Waxing Gibbous,55
2,Algeria,Algiers,36.76,3.05,Africa/Algiers,1715849100,2024-05-16 09:45,23.0,73.4,Sunny,...,10.4,18.4,1,1,05:40 AM,07:50 PM,01:15 PM,02:14 AM,Waxing Gibbous,55
3,Andorra,Andorra La Vella,42.50,1.52,Europe/Andorra,1715849100,2024-05-16 10:45,6.3,43.3,Light drizzle,...,0.7,0.9,1,1,06:31 AM,09:11 PM,02:12 PM,03:31 AM,Waxing Gibbous,55
4,Angola,Luanda,-8.84,13.23,Africa/Luanda,1715849100,2024-05-16 09:45,26.0,78.8,Partly cloudy,...,183.4,262.3,5,10,06:12 AM,05:55 PM,01:17 PM,12:38 AM,Waxing Gibbous,55


In [2]:
# Step 2: Basic info about the dataset
print("=== Dataset Info ===")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print("\n=== Column Names ===")
print(df.columns.tolist())
print("\n=== Data Types ===")
print(df.dtypes)
print("\n=== Missing Values ===")
print(df.isnull().sum())

=== Dataset Info ===
Rows: 144432
Columns: 41

=== Column Names ===
['country', 'location_name', 'latitude', 'longitude', 'timezone', 'last_updated_epoch', 'last_updated', 'temperature_celsius', 'temperature_fahrenheit', 'condition_text', 'wind_mph', 'wind_kph', 'wind_degree', 'wind_direction', 'pressure_mb', 'pressure_in', 'precip_mm', 'precip_in', 'humidity', 'cloud', 'feels_like_celsius', 'feels_like_fahrenheit', 'visibility_km', 'visibility_miles', 'uv_index', 'gust_mph', 'gust_kph', 'air_quality_Carbon_Monoxide', 'air_quality_Ozone', 'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10', 'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'sunrise', 'sunset', 'moonrise', 'moonset', 'moon_phase', 'moon_illumination']

=== Data Types ===
country                          object
location_name                    object
latitude                        float64
longitude                       float64
timezone                         o

In [3]:
# Step 3: Check duplicates and fix data types
print("=== Duplicate Rows ===")
print(f"Number of duplicates: {df.duplicated().sum()}")

# Convert last_updated to datetime (important for time series later!)
df['last_updated'] = pd.to_datetime(df['last_updated'])

# Extract useful time features
df['year'] = df['last_updated'].dt.year
df['month'] = df['last_updated'].dt.month
df['day'] = df['last_updated'].dt.day
df['hour'] = df['last_updated'].dt.hour

print("\n=== Date Range ===")
print(f"From: {df['last_updated'].min()}")
print(f"To:   {df['last_updated'].max()}")

print("\n=== Sample Countries ===")
print(df['country'].nunique(), "unique countries")
print(df['country'].unique()[:20])


=== Duplicate Rows ===
Number of duplicates: 0

=== Date Range ===
From: 2024-05-16 01:45:00
To:   2026-05-30 19:15:00

=== Sample Countries ===
211 unique countries
['Afghanistan' 'Albania' 'Algeria' 'Andorra' 'Angola'
 'Antigua and Barbuda' 'Argentina' 'Armenia' 'Australia' 'Austria'
 'Azerbaijan' 'Bahamas' 'Bahrain' 'Bangladesh' 'Barbados' 'Belarus'
 'Belgium' 'Belize' 'Benin' 'Bhutan']


In [4]:
# Step 4: Statistical summary and outlier detection
print("=== Statistical Summary ===")
df.describe().round(2)

=== Statistical Summary ===


,latitude,longitude,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,wind_mph,wind_kph,wind_degree,pressure_mb,...,air_quality_Sulphur_dioxide,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,moon_illumination,year,month,day,hour
count,144432.00,144432.00,1.444320e+05,144432,144432.00,144432.00,144432.00,144432.00,144432.00,144432.00,...,144432.00,144432.00,144432.00,144432.00,144432.00,144432.00,144432.00,144432.00,144432.00,144432.00
mean,19.22,21.93,1.747993e+09,2025-05-23 11:49:39.187438080,21.26,70.26,7.98,12.84,169.04,1014.06,...,10.08,23.80,47.70,1.68,2.57,49.83,2024.89,6.50,15.86,10.81
min,-41.30,-175.20,1.715849e+09,2024-05-16 01:45:00,-29.80,-21.60,2.20,3.60,1.00,947.00,...,-9999.00,0.17,-1848.15,1.00,1.00,0.00,2024.00,1.00,1.00,0.00
25%,4.05,-6.84,1.732006e+09,2024-11-19 05:00:00,15.70,60.20,3.80,6.10,80.00,1010.00,...,1.11,7.03,9.80,1.00,1.00,15.00,2024.00,4.00,8.00,8.00
50%,17.25,23.24,1.747991e+09,2025-05-23 12:00:00,23.70,74.60,6.70,10.80,161.00,1013.00,...,2.40,13.88,19.55,1.00,2.00,50.00,2025.00,6.00,16.00,11.00
75%,40.40,49.88,1.763968e+09,2025-11-24 13:52:30,27.90,82.30,11.00,17.60,256.00,1018.00,...,8.05,27.30,40.75,2.00,3.00,85.00,2025.00,9.00,23.00,14.00
max,64.15,179.22,1.780122e+09,2026-05-30 19:15:00,79.30,174.70,1841.20,2963.20,360.00,3006.00,...,521.33,1614.10,6037.29,6.00,10.00,100.00,2026.00,12.00,31.00,23.00
std,24.41,65.78,1.855475e+07,NaN,9.61,17.29,7.10,11.42,103.68,10.15,...,34.63,36.18,147.22,0.93,2.42,35.05,0.70,3.42,8.80,4.75


In [5]:
# Step 5: Remove outliers using IQR method
def remove_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return df[(df[column] >= lower) & (df[column] <= upper)]

# Apply to key columns
original_size = len(df)
df = remove_outliers_iqr(df, 'temperature_celsius')
df = remove_outliers_iqr(df, 'wind_mph')
df = remove_outliers_iqr(df, 'humidity')
df = remove_outliers_iqr(df, 'pressure_mb')

print(f"Rows before: {original_size}")
print(f"Rows after:  {len(df)}")
print(f"Removed:     {original_size - len(df)} outliers")

Rows before: 144432
Rows after:  136315
Removed:     8117 outliers


In [6]:
# Step 6: Normalize numerical columns and save clean data
from sklearn.preprocessing import MinMaxScaler

# Select numerical columns to normalize
num_cols = ['temperature_celsius', 'humidity', 'wind_mph', 
            'pressure_mb', 'precip_mm', 'visibility_km', 'uv_index']

# Create normalized versions (keep originals too)
scaler = MinMaxScaler()
df_scaled = df.copy()
df_scaled[num_cols] = scaler.fit_transform(df[num_cols])

# Save the clean dataset
df.to_csv('cleaned_weather_data.csv', index=False)
print("✅ Clean dataset saved as 'cleaned_weather_data.csv'")
print(f"Final shape: {df.shape}")
print("\nNormalized sample:")
df_scaled[num_cols].head(3).round(3)

✅ Clean dataset saved as 'cleaned_weather_data.csv'
Final shape: (136315, 45)

Normalized sample:


,temperature_celsius,humidity,wind_mph,pressure_mb,precip_mm,visibility_km,uv_index
0,0.599,0.224,0.313,0.438,0.000,0.312,0.429
1,0.442,0.939,0.241,0.438,0.002,0.312,0.307
2,0.525,0.276,0.369,0.406,0.000,0.312,0.307
